# omnicontext_check
Aggregates the OmniContext leg and sets it against the domain leg. Five runs at the pinned dataset revision: the 512 baseline every arm is measured against, three finalists at the engine settings of their domain runs, and the released-geometry calibration.

Reports pooled means over the 400 rows, which is the basis the domain headline uses, and the macro over the 8 subtasks, which is the basis the published leaderboard uses. Reads scores only; no GPU, no API calls.

## 1. Load

In [1]:
from google.colab import drive
import os, json, math, statistics as st
import numpy as np
from collections import defaultdict
from scipy import stats
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SC    = f"{DRIVE}/scores"
BASE  = "baseline"
ARMS  = ["lora_r8_step1000", "qlora_r8_step2000", "sp_iii_p139_step2000"]
CALIB = "baseline_shipped"

SIGMA_ITEM = 0.362        # judge re-scoring SD on a fixed image, RMS over sd_reps (90 rows x5)
BOUND      = 0.15         # pre-registered equivalence bound, TOST alpha 0.05
PUBLISHED  = {"single_character": 8.05, "single_object": 7.58,
              "multi_character": 7.11, "multi_object": 7.13, "multi_character_object": 7.45,
              "scene_character": 6.38, "scene_object": 6.71, "scene_character_object": 7.04}

ov = lambda d: math.sqrt(max(d["PF"], 0) * max(d["SC"], 0))

def load(path, key="i"):
    d, errs = {}, defaultdict(int)
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: errs[r["error"]] += 1; continue
        d[r[key]] = r
    return d, errs

OC = {}
for r in [BASE] + ARMS + [CALIB]:
    OC[r], e = load(f"{SC}/omnicontext__{r}.jsonl")
    print(f"omnicontext__{r:22s} {len(OC[r]):4d} rows" + (f"  errors {dict(e)}" if e else ""))

# the domain leg, for the cross-leg contrast; uid-keyed, as the domain scorers wrote it
DOM_OF = {"lora_r8_step1000": "test__lora_r8_step1000", "qlora_r8_step2000": "test__qlora_r8_step2000",
          "sp_iii_p139_step2000": "test__sp_iii_p139_step2000"}
DOM = {}
DOM["baseline"], _ = load(f"{SC}/test__baseline.jsonl", key="uid")
for a, f in DOM_OF.items():
    DOM[a], _ = load(f"{SC}/{f}.jsonl", key="uid")
print(f"\ndomain test__ files loaded: {len(DOM)} ({len(DOM['baseline'])} rows each)")

Mounted at /content/drive
omnicontext__baseline                400 rows
omnicontext__lora_r8_step1000        400 rows
omnicontext__qlora_r8_step2000       400 rows
omnicontext__sp_iii_p139_step2000    400 rows
omnicontext__baseline_shipped        400 rows

domain test__ files loaded: 4 (488 rows each)


## 2. Per-run levels
Macro is the mean of the eight subtask means, each of which is a mean of per-item geometric means: `calculate_statistics.py` verbatim. The published column is OmniGen2 at the released 1024x1024 `align_res=True` geometry, so it is the like-for-like reference for the calibration run and not for the 512 arms.

In [2]:
def macro(d):
    g = defaultdict(list)
    for r in d.values(): g[r["task_type"]].append(ov(r))
    return st.mean([st.mean(v) for v in g.values()]), {k: st.mean(v) for k, v in g.items()}

print(f"{'run':24s} {'n':>4s} {'PF':>6s} {'SC':>6s} {'pooled':>7s} {'macro':>7s} {'vs pub':>8s} {'Ovr=0':>6s}")
MAC = {}
for r in [BASE] + ARMS + [CALIB]:
    d = OC[r]; m, per = macro(d); MAC[r] = (m, per)
    z = sum(1 for x in d.values() if min(x["PF"], x["SC"]) <= 0)
    print(f"{r:24s} {len(d):4d} {st.mean([x['PF'] for x in d.values()]):6.3f} "
          f"{st.mean([x['SC'] for x in d.values()]):6.3f} {st.mean([ov(x) for x in d.values()]):7.3f} "
          f"{m:7.3f} {m-st.mean(list(PUBLISHED.values())):+8.3f} {z:6d}")

print(f"\n{'subtask':26s} {'pub':>6s} " + " ".join(f"{r[:12]:>12s}" for r in [BASE]+ARMS+[CALIB]))
for k in sorted(PUBLISHED):
    print(f"{k:26s} {PUBLISHED[k]:6.2f} " + " ".join(f"{MAC[r][1][k]:12.3f}" for r in [BASE]+ARMS+[CALIB]))

run                         n     PF     SC  pooled   macro   vs pub  Ovr=0
baseline                  400  7.575  7.433   7.343   7.343   +0.162     10
lora_r8_step1000          400  7.575  7.265   7.272   7.272   +0.091      9
qlora_r8_step2000         400  7.660  7.500   7.432   7.432   +0.251      7
sp_iii_p139_step2000      400  7.190  6.295   6.528   6.528   -0.653     17
baseline_shipped          400  7.537  6.870   7.015   7.015   -0.166     13

subtask                       pub     baseline lora_r8_step qlora_r8_ste sp_iii_p139_ baseline_shi
multi_character              7.11        6.921        6.848        7.141        5.791        6.255
multi_character_object       7.45        7.710        7.515        7.673        6.584        7.376
multi_object                 7.13        7.566        7.587        7.471        7.339        7.039
scene_character              6.38        6.928        6.941        7.036        5.613        6.336
scene_character_object       7.04        7.186  

## 3. Paired against the 512 baseline
Every row is the same benchmark case under both runs, so the contrast is paired. Alignment is asserted on `key`; a row index alone would not catch two runs scored from different image sets.

In [3]:
def paired(a, b, fn=ov):
    I = sorted(set(a) & set(b))
    bad = [i for i in I if a[i]["key"] != b[i]["key"]]
    assert not bad, f"key mismatch on rows {bad[:5]} - runs are not aligned"
    d = np.array([fn(a[i]) - fn(b[i]) for i in I], float)
    n = len(d); se = d.std(ddof=1) / math.sqrt(n)
    t, p = stats.ttest_1samp(d, 0)
    return dict(n=n, mean=d.mean(), sd=d.std(ddof=1), se=se, t=t, p=p,
                lo=d.mean() - stats.t.ppf(0.95, n-1)*se, hi=d.mean() + stats.t.ppf(0.95, n-1)*se,
                mde=(stats.norm.ppf(0.975)+stats.norm.ppf(0.80))*se, I=I)

def tost(r, bound=BOUND):
    df = r["n"] - 1
    p_lo = 1 - stats.t.cdf((r["mean"] + bound)/r["se"], df)
    p_hi = stats.t.cdf((r["mean"] - bound)/r["se"], df)
    return max(p_lo, p_hi)

print(f"{'contrast':26s} {'metric':8s} {'mean':>7s} {'90% CI':>18s} {'p':>8s} {'MDE':>7s} {'TOST':>7s}")
for r in ARMS + [CALIB]:
    for lab, fn in [("PF", lambda d: d["PF"]), ("SC", lambda d: d["SC"]), ("Overall", ov)]:
        s = paired(OC[r], OC[BASE], fn)
        tp = f"{tost(s):.3f}" if lab == "Overall" else ""
        print(f"{r[:26]:26s} {lab:8s} {s['mean']:+7.3f} [{s['lo']:+7.3f},{s['hi']:+7.3f}] "
              f"{s['p']:8.4f} {s['mde']:7.3f} {tp:>7s}")
    print()

contrast                   metric      mean             90% CI        p     MDE    TOST
lora_r8_step1000           PF        +0.000 [ -0.111, +0.111]   1.0000   0.188        
lora_r8_step1000           SC        -0.168 [ -0.284, -0.051]   0.0179   0.197        
lora_r8_step1000           Overall   -0.071 [ -0.166, +0.025]   0.2216   0.162   0.085

qlora_r8_step2000          PF        +0.085 [ -0.037, +0.207]   0.2517   0.207        
qlora_r8_step2000          SC        +0.068 [ -0.048, +0.183]   0.3371   0.197        
qlora_r8_step2000          Overall   +0.089 [ -0.012, +0.191]   0.1468   0.172   0.162

sp_iii_p139_step2000       PF        -0.385 [ -0.577, -0.193]   0.0011   0.327        
sp_iii_p139_step2000       SC        -1.137 [ -1.328, -0.947]   0.0000   0.323        
sp_iii_p139_step2000       Overall   -0.815 [ -0.989, -0.641]   0.0000   0.295   1.000

baseline_shipped           PF        -0.037 [ -0.250, +0.175]   0.7710   0.361        
baseline_shipped           SC        -0

## 4. Cross-leg
The domain split is 50% single-reference against 25% here, and difficulty falls sharply with reference count, so pooled numbers do not transfer between legs. The delta does: each arm is measured against its own leg's baseline, and the two deltas answer the same question on different content.

In [4]:
def paired_uid(a, b, fn=ov):
    I = sorted(set(a) & set(b))
    d = np.array([fn(a[i]) - fn(b[i]) for i in I], float)
    n = len(d); se = d.std(ddof=1)/math.sqrt(n)
    return dict(n=n, mean=d.mean(), se=se, p=stats.ttest_1samp(d, 0)[1])

print(f"{'arm':24s} {'domain (488)':>22s} {'omnicontext (400)':>24s} {'gap':>16s}")
for a in ARMS:
    o = paired(OC[a], OC[BASE]); dm = paired_uid(DOM[a], DOM["baseline"])
    z = (o["mean"] - dm["mean"]) / math.sqrt(o["se"]**2 + dm["se"]**2)
    pz = 2*(1 - stats.norm.cdf(abs(z)))
    print(f"{a:24s} {dm['mean']:+9.3f} (p {dm['p']:.3f}) {o['mean']:+11.3f} (p {o['p']:.3f}) "
          f"{o['mean']-dm['mean']:+8.3f} ({z:+.2f} SE, p {pz:.3f})")
print("\n  a gap with no statistically detectable difference from zero is not evidence that the domain and OmniContext")
print("  effects differ; any significantly non-zero gap would support heterogeneity across legs,")
print("  with the sign indicating which leg carries the larger cost")

arm                                domain (488)        omnicontext (400)              gap
lora_r8_step1000            -0.120 (p 0.027)      -0.071 (p 0.222)   +0.049 (+0.62 SE, p 0.534)
qlora_r8_step2000           -0.080 (p 0.200)      +0.089 (p 0.147)   +0.169 (+1.93 SE, p 0.053)
sp_iii_p139_step2000        -0.826 (p 0.000)      -0.815 (p 0.000)   +0.011 (+0.08 SE, p 0.936)

  a gap with no statistically detectable difference from zero is not evidence that the domain and OmniContext
  effects differ; any significantly non-zero gap would support heterogeneity across legs,
  with the sign indicating which leg carries the larger cost


## 5. Strata
Reference count and the scene flag are properties of the benchmark row, fixed before generation, so both cuts are legitimate. `with_scene` also selects the judging rubric.

In [5]:
print("baseline level by stratum:")
for key, order in [("n_refs", [1,2,3]), ("with_scene", [False, True])]:
    g = defaultdict(list)
    for x in OC[BASE].values(): g[x[key]].append(ov(x))
    print(f"  {key:11s} " + " | ".join(f"{k}: {st.mean(g[k]):.3f} (n={len(g[k])})" for k in order))

for key, order in [("n_refs", [1,2,3]), ("with_scene", [False, True])]:
    print(f"\nOverall delta vs baseline by {key}:")
    print(f"  {'run':24s} " + " ".join(f"{str(k):>16s}" for k in order))
    for r in ARMS + [CALIB]:
        I = paired(OC[r], OC[BASE])["I"]
        cells = []
        for k in order:
            u = [i for i in I if OC[r][i][key] == k]
            d = np.array([ov(OC[r][i]) - ov(OC[BASE][i]) for i in u])
            cells.append(f"{d.mean():+.3f} p{stats.ttest_1samp(d,0)[1]:.3f}")
        print(f"  {r:24s} " + " ".join(f"{c:>16s}" for c in cells))

print("\nOverall delta vs baseline by subtask:")
print(f"  {'subtask':26s} " + " ".join(f"{r[:14]:>14s}" for r in ARMS + [CALIB]))
for k in sorted(PUBLISHED):
    row = []
    for r in ARMS + [CALIB]:
        I = [i for i in paired(OC[r], OC[BASE])["I"] if OC[r][i]["task_type"] == k]
        row.append(f"{st.mean([ov(OC[r][i]) - ov(OC[BASE][i]) for i in I]):+.3f}")
    print(f"  {k:26s} " + " ".join(f"{c:>14s}" for c in row))

baseline level by stratum:
  n_refs      1: 7.655 (n=100) | 2: 7.462 (n=210) | 3: 6.720 (n=90)
  with_scene  False: 7.501 (n=250) | True: 7.079 (n=150)

Overall delta vs baseline by n_refs:
  run                                     1                2                3
  lora_r8_step1000            -0.128 p0.347    -0.062 p0.413    -0.027 p0.809
  qlora_r8_step2000           +0.128 p0.235    +0.077 p0.405    +0.075 p0.531
  sp_iii_p139_step2000        -0.707 p0.001    -0.924 p0.000    -0.682 p0.001
  baseline_shipped            +0.053 p0.841    -0.478 p0.003    -0.399 p0.061

Overall delta vs baseline by with_scene:
  run                                 False             True
  lora_r8_step1000            -0.101 p0.144    -0.020 p0.843
  qlora_r8_step2000           +0.069 p0.322    +0.124 p0.291
  sp_iii_p139_step2000        -0.779 p0.000    -0.874 p0.000
  baseline_shipped            -0.284 p0.048    -0.400 p0.052

Overall delta vs baseline by subtask:
  subtask                    lora_

## 6. Geometry
`baseline_shipped` shares weights, sampler, seed, negative prompt and hardware with the 512 baseline and differs only in output size, `align_res` and reference downscaling. The contrast is therefore the engine-geometry effect, and it is the run to compare against the published average.

In [6]:
for r in [BASE, CALIB]:
    p = f"{DRIVE}/gen/omnicontext/{r}/manifest.json"
    if not os.path.exists(p): continue
    m = json.load(open(p)); g = m.get("geometry", {})
    print(f"{r:18s} {g.get('width')}x{g.get('height')} align_res={g.get('align_res')} | "
          f"{m.get('gpu')} | torch {m.get('torch')}")
    if g.get("output_sizes_seen"):
        s = g["output_sizes_seen"]
        print(f"{'':18s} {len(s)} distinct output sizes; align_res applies only to single-reference rows")

s = paired(OC[CALIB], OC[BASE])
print(f"\nreleased geometry minus 512: Overall {s['mean']:+.3f} (p {s['p']:.4f})")
print(f"  macro: released {MAC[CALIB][0]:.3f} vs published {st.mean(list(PUBLISHED.values())):.3f} "
      f"({MAC[CALIB][0]-st.mean(list(PUBLISHED.values())):+.3f}) | 512 baseline {MAC[BASE][0]:.3f}")
print("  the released-geometry run is the like-for-like comparison against the published average;")
print("  it remains 0.166 below the published macro (7.015 vs 7.181), so exact numerical")
print("  reproduction was not achieved and the residual is unexplained.")

baseline           512x512 align_res=False | NVIDIA A100-SXM4-40GB | torch 2.6.0+cu124
baseline_shipped   1024x1024 align_res=True | NVIDIA A100-SXM4-40GB | torch 2.6.0+cu124
                   22 distinct output sizes; align_res applies only to single-reference rows

released geometry minus 512: Overall -0.328 (p 0.0056)
  macro: released 7.015 vs published 7.181 (-0.166) | 512 baseline 7.343
  the released-geometry run is the like-for-like comparison against the published average;
  it remains 0.166 below the published macro (7.015 vs 7.181), so exact numerical
  reproduction was not achieved and the residual is unexplained.


## 7. Resolution

In [7]:
floor = math.sqrt(2) * SIGMA_ITEM
print(f"judge-only floor on a paired delta: sqrt(2)*{SIGMA_ITEM} = {floor:.3f} per row")
print(f"{'run':24s} {'sigma_d':>8s} {'SE':>7s} {'MDE':>7s} {'verdict at the pre-registered bound':>38s}")
for r in ARMS + [CALIB]:
    s = paired(OC[r], OC[BASE]); t = tost(s)
    if s["p"] < 0.05 and abs(s["mean"]) > BOUND: v = "significant and beyond the bound"
    elif t < 0.05:                               v = f"equivalent within +/-{BOUND}"
    elif s["p"] < 0.05:                          v = "significant, magnitude inside the bound"
    else:                                        v = "inconclusive: neither effect nor equivalence"
    print(f"{r:24s} {s['sd']:8.3f} {s['se']:7.3f} {s['mde']:7.3f} {v:>38s}")
print("\nMDE is arm-specific; the adapter legs here are no noisier than on the domain leg:")
for r in ARMS:
    s = paired(OC[r], OC[BASE])
    print(f"  {r:24s} sigma_d {s['sd']:.3f} -> effects below {s['mde']:.3f} Overall "
          f"are tested with under 80% power at n=400")

judge-only floor on a paired delta: sqrt(2)*0.362 = 0.512 per row
run                       sigma_d      SE     MDE    verdict at the pre-registered bound
lora_r8_step1000            1.154   0.058   0.162 inconclusive: neither effect nor equivalence
qlora_r8_step2000           1.229   0.061   0.172 inconclusive: neither effect nor equivalence
sp_iii_p139_step2000        2.108   0.105   0.295       significant and beyond the bound
baseline_shipped            2.354   0.118   0.330       significant and beyond the bound

MDE is arm-specific; the adapter legs here are no noisier than on the domain leg:
  lora_r8_step1000         sigma_d 1.154 -> effects below 0.162 Overall are tested with under 80% power at n=400
  qlora_r8_step2000        sigma_d 1.229 -> effects below 0.172 Overall are tested with under 80% power at n=400
  sp_iii_p139_step2000     sigma_d 2.108 -> effects below 0.295 Overall are tested with under 80% power at n=400
